# Введение в MapReduce модель на Python


In [44]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [45]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [46]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [47]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.2910612487164563)),
 (1, np.float64(2.2910612487164563)),
 (2, np.float64(2.2910612487164563)),
 (3, np.float64(2.2910612487164563)),
 (4, np.float64(2.2910612487164563))]

## Inverted index

In [48]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [49]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('what', 10)]),
 (1, [('a', 2), ('it', 18)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.005048156396695425)),
   (None, np.float64(0.04263467892104522)),
   (None, np.float64(0.11863702727500414)),
   (None, np.float64(0.1314874115788074)),
   (None, np.float64(0.13950878899118824)),
   (None, np.float64(0.23255480987028343)),
   (None, np.float64(0.2899093194562333)),
   (None, np.float64(0.3287426370253316)),
   (None, np.float64(0.3681219786563815)),
   (None, np.float64(0.38432555656330547)),
   (None, np.float64(0.4049261520775057)),
   (None, np.float64(0.4470684230633324)),
   (None, np.float64(0.4498836605688794)),
   (None, np.float64(0.47071668435866565)),
   (None, np.float64(0.49507466669472244))]),
 (1,
  [(None, np.float64(0.535424151176426)),
   (None, np.float64(0.5372408277501078)),
   (None, np.float64(0.5634217024968925)),
   (None, np.float64(0.588611998089516)),
   (None, np.float64(0.5939484919287047)),
   (None, np.float64(0.5958235541842368)),
   (None, np.float64(0.6771187382594482)),
   (None, np.float64(0.698595582902

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
import numpy as np
from typing import Iterator, NamedTuple

input_numbers = np.random.rand(20) * 100  # 20 случайных чисел

def RECORDREADER():
    for i, value in enumerate(input_numbers):
        yield (i, value)

def MAP(_, value: float):
    yield ("max", value)

def REDUCE(key: str, values: Iterator[float]):
    max_value = max(values)
    yield (key, max_value)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Входные числа: {input_numbers}")
print(f"Максимальное значение: {output}")

Входные числа: [39.01222157 18.85486833 51.25798798 70.18147217 85.66489681 82.19672421
 25.82443129 73.18530354 65.10886    76.93274175 78.76141418 64.98629768
 17.20519818 64.22405851 10.69874842 96.04403723 63.6484171  25.5121507
 62.37670894 29.90613286]
Максимальное значение: [('max', np.float64(96.04403723088465))]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
import numpy as np
from typing import Iterator

input_numbers = np.random.rand(20) * 100

def RECORDREADER():
    for i, value in enumerate(input_numbers):
        yield (i, value)

def MAP(_, value: float):
    yield ("sum_count", (value, 1))

def REDUCE(key: str, values: Iterator[tuple]):
    total_sum = 0
    total_count = 0
    for value, count in values:
        total_sum += value
        total_count += count

    if total_count > 0:
        mean = total_sum / total_count
        yield (key, mean)
    else:
        yield (key, 0)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Входные числа: {input_numbers}")
print(f"Арифметическое среднее: {output}")
print(f"Проверка с помощью numpy: {np.mean(input_numbers)}")

Входные числа: [38.58577853 35.13485449  0.1948237  91.58479053 90.21736527 35.44226777
 29.14661117 47.33826108  8.55286959  9.05755553  5.82969928 78.54673235
 83.42590208 36.95036896 80.29851343 73.09797779 18.9017232  56.27589562
 64.07915275 80.3836634 ]
Арифметическое среднее: [('sum_count', np.float64(48.152240326560914))]
Проверка с помощью numpy: 48.152240326560914


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sort(iterable):
    sorted_items = sorted(iterable, key=lambda x: x[0])

    result = []
    current_key = None
    current_values = []

    for key, value in sorted_items:
        if key != current_key:
            if current_key is not None:
                result.append((current_key, current_values))
            current_key = key
            current_values = [value]
        else:
            current_values.append(value)

    if current_key is not None:
        result.append((current_key, current_values))

    return result

test_data = [('b', 2), ('a', 1), ('c', 3), ('b', 4), ('a', 5)]
result = groupbykey_sort(test_data)
print(f"Исходные данные: {test_data}")
print(f"Результат groupbykey_sort: {result}")

Исходные данные: [('b', 2), ('a', 1), ('c', 3), ('b', 4), ('a', 5)]
Результат groupbykey_sort: [('a', [1, 5]), ('b', [2, 4]), ('c', [3])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
import numpy as np
from typing import Iterator

data_with_duplicates = [1, 2, 3, 2, 4, 1, 5, 3, 6, 7, 5, 8]

def RECORDREADER():
    for i, value in enumerate(data_with_duplicates):
        yield (i, value)

def MAP(_, value):
    yield (value, None)

def REDUCE(key, _):
    yield (key, None)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
unique_elements = [item[0] for item in output]
print(f"Исходные данные: {data_with_duplicates}")
print(f"Уникальные элементы: {unique_elements}")
print(f"С помощью set: {set(data_with_duplicates)}")

Исходные данные: [1, 2, 3, 2, 4, 1, 5, 3, 6, 7, 5, 8]
Уникальные элементы: [1, 2, 3, 4, 5, 6, 7, 8]
С помощью set: {1, 2, 3, 4, 5, 6, 7, 8}


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [51]:
class Person(NamedTuple):
    id: int
    name: str
    age: int
    city: str

people = [
    Person(id=1, name="Vika", age=22, city="Samara"),
    Person(id=2, name="Anna", age=23, city="Saint Petersburg"),
    Person(id=3, name="Ira", age=19, city="Moscow"),
    Person(id=4, name="Polina", age=25, city="Samara"),
    Person(id=5, name="Nastya", age=20, city="Samara")
]

def RECORDREADER():
    for person in people:
        yield (person.id, person)

def MAP(_, person: Person):
    if person.age > 20 and person.city == "Samara":
        yield (person, person)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
selected = [item[0] for item in output]
print("Исходные данные:")
for person in people:
    print(f"  {person}")
print("\nВыбранные люди (возраст > 20 и город Самара):")
for person in selected:
    print(f"  {person}")

Исходные данные:
  Person(id=1, name='Vika', age=22, city='Samara')
  Person(id=2, name='Anna', age=23, city='Saint Petersburg')
  Person(id=3, name='Ira', age=19, city='Moscow')
  Person(id=4, name='Polina', age=25, city='Samara')
  Person(id=5, name='Nastya', age=20, city='Samara')

Выбранные люди (возраст > 20 и город Самара):
  Person(id=1, name='Vika', age=22, city='Samara')
  Person(id=4, name='Polina', age=25, city='Samara')


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [53]:
def RECORDREADER():
    for person in people:
        yield (person.id, person)

def MAP(_, person: Person):
    projected = (person.name, person.age)
    yield (projected, projected)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
projections = [item[0] for item in output]
print("Исходные данные:")
for person in people:
    print(f"  {person}")
print("\nПроекция на (имя, возраст) с уникальными значениями:")
for p in projections:
    print(f"  {p}")

Исходные данные:
  Person(id=1, name='Vika', age=22, city='Samara')
  Person(id=2, name='Anna', age=23, city='Saint Petersburg')
  Person(id=3, name='Ira', age=19, city='Moscow')
  Person(id=4, name='Polina', age=25, city='Samara')
  Person(id=5, name='Nastya', age=20, city='Samara')

Проекция на (имя, возраст) с уникальными значениями:
  ('Vika', 22)
  ('Anna', 23)
  ('Ira', 19)
  ('Polina', 25)
  ('Nastya', 20)


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [59]:
set1 = [(1,"Vika"), (2,"Ira"), (3,"Nastya")]
set2 = [(4,"Anna"), (5,"Polina")]

def RECORDREADER():
    for value in set1:
        yield (f"set1_{value}", value)
    for value in set2:
        yield (f"set2_{value}", value)

def MAP(_, value):
    yield (value, value)

def REDUCE(key, values):
    yield (key, key)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([s for (_, s) in output])

[(1, 'Vika'), (2, 'Ira'), (3, 'Nastya'), (4, 'Anna'), (5, 'Polina')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [60]:
set1 = [(1,"Vika"), (2,"Ira"), (3,"Nastya")]
set2 = [(3,"Nastya"), (4,"Polina")]

def RECORDREADER():
    for value in set1:
        yield (f"set1_{value}", value)
    for value in set2:
        yield (f"set2_{value}", value)

def MAP(_, value):
    yield (value, value)

def REDUCE(t, values):
    vals = list(values)
    if len(vals) == 2:
        yield (t, t)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([s for (_, s) in output])

[(3, 'Nastya')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [63]:
set1 = [(1,"Vika"), (2,"Ira"), (3,"Nastya")]
set2 = [(3,"Nastya"), (4,"Polina")]
def RECORDREADER():
    for t in set1:
        yield ("set1", t)
    for t in set2:
        yield ("set2", t)

def MAP(source, t):
    yield (t, source)

def REDUCE(t, sources):
    srcs = list(sources)
    if srcs == ["set1"]:
        yield (t, t)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([s for (_, s) in output])

[(1, 'Vika'), (2, 'Ira')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [65]:
set1 = [(1, 'A'), (2, 'B'), (3, 'C'), (4, 'D')]
set2 = [('A', 'X'), ('B', 'Y'), ('B', 'Z'), ('E', 'W')]

def RECORDREADER():
    for a, b in set1:
        yield ((a, b), ('set1', a))
    for b, c in set2:
        yield ((b, c), ('set2', c))

def MAP(key, value):
    relation, attr = value
    if relation == 'set1':
        b = key[1]
        yield (b, ('set1', attr))
    else:
        b = key[0]
        yield (b, ('set2', attr))

def REDUCE(key, values):
    values_list = list(values)
    r_values = [v[1] for v in values_list if v[0] == 'set1']
    s_values = [v[1] for v in values_list if v[0] == 'set2']

    for a in r_values:
        for c in s_values:
            yield (None, (a, key, c))

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([s for (_, s) in output])

[(1, 'A', 'X'), (2, 'B', 'Y'), (2, 'B', 'Z')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [67]:
data = [
    ('A', 10), ('B', 20), ('A', 15), ('C', 5),
    ('B', 25), ('A', 30), ('C', 10), ('B', 15)
]

def RECORDREADER():
    for i, (group, value) in enumerate(data):
        yield (i, (group, value))

def MAP(_, record):
    group, value = record
    yield (group, value)

def REDUCE(group, values):
    values_list = list(values)
    total = sum(values_list)
    count = len(values_list)
    mean = total / count if count > 0 else 0
    maximum = max(values_list)
    minimum = min(values_list)

    yield (group, {
        'sum': total,
        'count': count,
        'mean': mean,
        'max': maximum,
        'min': minimum
    })

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Исходные данные:")
for group, value in data:
    print(f"  {group}: {value}")
print("\nРезультаты агрегации по группам:")
for group, agg in output:
    print(f"  Группа {group}: {agg}")

Исходные данные:
  A: 10
  B: 20
  A: 15
  C: 5
  B: 25
  A: 30
  C: 10
  B: 15

Результаты агрегации по группам:
  Группа A: {'sum': 55, 'count': 3, 'mean': 18.333333333333332, 'max': 30, 'min': 10}
  Группа B: {'sum': 60, 'count': 3, 'mean': 20.0, 'max': 25, 'min': 15}
  Группа C: {'sum': 15, 'count': 2, 'mean': 7.5, 'max': 10, 'min': 5}


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [80]:
A = [
     (0,0,5), (0,2,3),
    (1,1,2), (1,3,7)
]

x = [
    (0,10),
    (1,20)
]

def RECORDREADER():
    for (i,j,v) in A:
        yield ("A", (i,j,v))
    for (j,v) in x:
        yield ("X", (j,v))

def MAP(source, t):
    if source == "A":
        i,j,v = t
        yield (j, ("A", i, v))
    else:
        j,v = t
        yield (j, ("X", v))

def REDUCE(j, values):
    A_vals = []
    X_vals = []

    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, x_val = val
            X_vals.append(x_val)

    for (i, a_val) in A_vals:
        for x_val in X_vals:
            yield (i, a_val * x_val)

stage1 = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    for (i,val) in stage1:
        yield (i,val)

def MAP2(i,val):
    yield (i,val)

def REDUCE2(i, values):
    yield (i, sum(values))

stage2 = MapReduce(RECORDREADER2, MAP2, REDUCE2)

print(list(stage2))

[(0, 50), (1, 40)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [73]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [74]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    total = 0.0
    for v in values:
        total += v
    yield (key, total)

Проверьте своё решение

In [75]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [76]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [81]:
A = [
    (0,0,1), (0,1,2), (0,2,3),
    (1,0,4), (1,1,5), (1,2,6)
]

B = [
    (0,0,9), (0,1,8), (0,2,7),
    (1,0,6), (1,1,5), (1,2,4)
]

def RECORDREADER():
    for (i,k,v) in A:
        yield ("A", (i,k,v))
    for (k,j,v) in B:
        yield ("B", (k,j,v))

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []

    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)


intermediate = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    for (key,val) in intermediate:
        yield (key,val)

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

output = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))
print(sorted(output))

[((0, 0), 21), ((0, 1), 18), ((0, 2), 15), ((1, 0), 66), ((1, 1), 57), ((1, 2), 48)]


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [82]:
A = [
    (0,0,1), (0,1,2), (0,2,3),
    (1,0,4), (1,1,5), (1,2,6)
]

B = [
    (0,0,9), (0,1,8), (0,2,7),
    (1,0,6), (1,1,5), (1,2,4)
]

maps = 2
reducers = 2

def INPUTFORMAT():
    def RECORDREADER_A():
        for t in A:
            yield ("A", t)
    def RECORDREADER_B():
        for t in B:
            yield ("B", t)
    yield RECORDREADER_A()
    yield RECORDREADER_B()

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []
    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)

stage1 = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

intermediate = []
for (_, partition) in stage1:
    intermediate.extend(list(partition))

def INPUTFORMAT2():
    def RR():
        for (key,val) in intermediate:
            yield (key,val)
    yield RR()

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)
result = []
for (_, partition) in stage2:
    result.extend(list(partition))

print(sorted(result))

12 key-value pairs were sent over a network.
12 key-value pairs were sent over a network.
[((0, 0), 21), ((0, 1), 18), ((0, 2), 15), ((1, 0), 66), ((1, 1), 57), ((1, 2), 48)]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [84]:
A = [
    (0,0,1), (0,1,2), (0,2,3),
    (1,0,4), (1,1,5), (1,2,6)
]

B = [
    (0,0,9), (0,1,8), (0,2,7),
    (1,0,6), (1,1,5), (1,2,4)
]

maps = 4
reducers = 2

def INPUTFORMAT():
    splitA = np.array_split(A, 2)
    splitB = np.array_split(B, 2)

    for chunk in splitA:
        def RR_A(chunk=chunk):
            for t in chunk:
                yield ("A", tuple(t))
        yield RR_A()

    for chunk in splitB:
        def RR_B(chunk=chunk):
            for t in chunk:
                yield ("B", tuple(t))
        yield RR_B()

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []
    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)

stage1 = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

intermediate = []
for (_, partition) in stage1:
    intermediate.extend(list(partition))

def INPUTFORMAT2():
    def RR():
        for (key,val) in intermediate:
            yield (key,val)
    yield RR()

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)

result = []
for (_, partition) in stage2:
    result.extend(list(partition))

print(sorted(result))

12 key-value pairs were sent over a network.
12 key-value pairs were sent over a network.
[((np.int64(0), np.int64(0)), np.int64(21)), ((np.int64(0), np.int64(1)), np.int64(18)), ((np.int64(0), np.int64(2)), np.int64(15)), ((np.int64(1), np.int64(0)), np.int64(66)), ((np.int64(1), np.int64(1)), np.int64(57)), ((np.int64(1), np.int64(2)), np.int64(48))]
